# 📊 Credit Risk Scoring: Exploratory Data Analysis (EDA)

**Business Context**: Оценка вероятности дефолта заемщика (просрочка свыше 90 дней за 2 года) для оптимизации кредитного портфеля банка.

**Цели анализа**:
1. Оценить баланс классов и обосновать выбор метрик (**ROC-AUC**, **Gini**, **PR-AUC** вместо Accuracy).
2. Исследовать распределения и пропуски в ключевых банковских атрибутах (`MonthlyIncome`, `DebtRatio`, `RevolvingUtilization`).
3. Сформировать доменные гипотезы для Feature Engineering.

In [ ]:
import yaml
import numpy as np
import pandas as pd

with open('../configs/config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

train_path = '../' + config['data']['processed_train_path']
df = pd.read_parquet(train_path)
print(f"Train observations: {df.shape[0]}, Features: {df.shape[1]}")
df.head()

## 1. Баланс целевой переменной (`SeriousDlqin2yrs`)

In [ ]:
target_dist = df['SeriousDlqin2yrs'].value_counts(normalize=True) * 100
print('Распределение классов (%):')
print(target_dist.round(2))

default_rate = df['SeriousDlqin2yrs'].mean()
print(f'\nБазовый уровень дефолта (Default Rate): {default_rate:.4f} ({default_rate*100:.2f}%)')

**Вывод по таргету:**
- Доля дефолтов всего ~6.68% (дисбаланс ~ 1:14).
- Метрика `Accuracy` вводит в заблуждение (константный классификатор дает 93.3% accuracy).
- Будем оптимизировать **ROC-AUC** и **PR-AUC**, а итоговое качество измерять банковским коэффициентом **Gini** ($2 \times \text{AUC} - 1$).

## 2. Анализ пропущенных значений

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0]

## 3. Банковская специфика и Feature Engineering идеи
- `MonthlyIncome` пропущен у ~19.7% заемщиков. Заемщики без декларированного дохода требуют создания флага `MonthlyIncome_is_missing`.
- `NumberOfDependents` пропущен у ~2.6% клиентов.
- Просрочки (`30-59`, `60-89`, `90+` дней): их сумма дает `TotalLateCount`.
- Финансовая нагрузка на члена семьи: `IncomePerPerson = MonthlyIncome / (NumberOfDependents + 1)`.
- Превышение лимита: `UtilizationOverLimit = RevolvingUtilizationOfUnsecuredLines > 1.0`.